# Notebook 08: Prefix & Prompt Caching

## Why This Matters
In real deployments, many requests share identical prefixes: the same system prompt, few-shot examples, or document context. Without caching, every request recomputes the KV cache for these shared tokens from scratch — wasting both compute and time. Prefix caching stores computed KV states and reuses them across requests, cutting Time-to-First-Token (TTFT) for long shared prompts from seconds to milliseconds.

This is used in production by OpenAI ("Prompt Caching"), Anthropic ("Prompt Caching"), Google ("Context Caching"), and vLLM's automatic prefix caching feature.

## Structure
1. The repeated-prefix problem
2. Hash-based cache lookup
3. Implementing a prefix cache
4. TTFT savings calculation
5. Cache eviction policies (LRU)
6. When prefix caching helps and when it doesn't
7. API-level prompt caching (OpenAI / Anthropic)

## What You'll Learn
- How to identify cacheable prefixes using content hashing
- The exact TTFT and cost savings for different prefix lengths
- LRU cache eviction and hit rate analysis
- How to structure prompts to maximize cache hit rate

## Background

### The repeated-computation problem

In real deployments, a large fraction of every request is identical. A customer service chatbot sends the same 2000-token system prompt with every request. A RAG application re-attaches the same retrieved document to hundreds of questions. A code assistant includes the same large file in every query. A multi-turn conversation grows by one user message per turn, but the preceding turns must be re-processed each time.

Without prefix caching, every request recomputes the KV cache for these shared tokens from scratch. The compute is wasted; the latency is paid by the user. For a 2000-token system prompt, this is the equivalent of throwing away 97% of the work on every request (assuming 50-token average user messages).

### The caching insight

The transformer's KV cache is **deterministic given the same input tokens and model weights**. This means the KV state for a shared prefix is identical across all requests that share it. We can compute it once, store it, and reuse it — the exact same values — for every subsequent request.

This is prefix caching. The implementation, in the context of PagedAttention, is straightforward: hash each KV block by its token IDs (chained so the hash is position-aware), store completed blocks in a hash table, and on each new request, look up how many prefix blocks are already cached. Skip the prefill for those tokens entirely.

### Why chained hashing?

A naive approach would hash each block of tokens independently. But this creates a collision risk: the block `[531, 8921, 42, 7777]` at position 0 would have the same hash as those same tokens at position 16. That would be a bug — the KV values at different positions are different because positional encodings differ.

Chained hashing solves this: each block's hash incorporates the previous block's hash. Block 0's hash depends only on its tokens. Block 1's hash depends on block 0's hash and its own tokens. This makes the hash path-dependent — the same tokens at different positions get different hashes. vLLM, Anthropic's API, and OpenAI's API all use this approach.

### Industry adoption

Prefix caching went from a research idea to a product feature across the major providers in 2023–2024:

- **vLLM** added automatic prefix caching (APC) — no code changes needed, hit rate is tracked per session
- **OpenAI** launched automatic prompt caching in late 2023 — prompts over 1024 tokens get automatic 50% discount on cached tokens, no opt-in required
- **Anthropic** launched explicit prompt caching in 2024 — developers mark cacheable sections with `cache_control` headers, cached reads cost 10% of normal price
- **Google** launched context caching for Gemini — explicit API with named cache objects and per-session billing

For high-volume production applications with long shared prefixes, prompt caching typically reduces input token costs by 60–90% and TTFT by 10–100×. It's one of the highest-leverage optimizations available without any model changes.

In [ ]:
%pip install -q torch

In [ ]:
import hashlib
import time
import collections
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import torch

print("Imports OK")

## 1. The Repeated-Prefix Problem

Common deployment patterns that create repeated prefixes:

```
Pattern 1: System prompt
  Request 1: [system_prompt][user_msg_1] → response
  Request 2: [system_prompt][user_msg_2] → response  ← same prefix!
  Request 3: [system_prompt][user_msg_3] → response  ← same prefix!

Pattern 2: Few-shot examples
  All requests: [few_shot_examples][new_question]
  The few-shot examples are identical across all requests

Pattern 3: RAG (document context)
  All requests about same doc: [retrieved_doc][question]
  If many questions about same document, doc context is recomputed each time

Pattern 4: Multi-turn chat
  Turn 1: [system][msg1]
  Turn 2: [system][msg1][resp1][msg2]  ← system+msg1 already computed in turn 1
  Turn 3: [system][msg1][resp1][msg2][resp2][msg3]
```

In each case, prefix computation is repeated work. For a 2000-token system prompt, this means rerunning a 2000-token prefill for every request.

In [ ]:
# Quantify the wasted compute

def prefill_time_ms(n_tokens: int, model_flops_per_token_per_layer: float = 2e9,
                    n_layers: int = 32, gpu_tflops: float = 312.0) -> float:
    """
    Estimate prefill time in ms.
    A100 BF16 peak: 312 TFLOPS
    LLaMA-7B: ~2 GFLOP per token per layer (rough estimate)
    """
    total_flops = n_tokens * model_flops_per_token_per_layer * n_layers
    return total_flops / (gpu_tflops * 1e12) * 1000  # ms


scenarios = [
    ("Short system prompt",     200,  50),
    ("Long system prompt",     2000, 200),
    ("RAG document context",   5000, 100),
    ("Few-shot examples",       500, 500),
    ("Claude context window",  4096, 1000),
]

print("Wasted prefill compute from repeated prefix processing:")
print(f"{'Scenario':<28} {'Prefix':>8} {'Requests':>10} {'Total wasted (ms)':>20} {'Total wasted (s)':>18}")
print("-" * 90)
for name, prefix_len, n_requests in scenarios:
    per_request_ms = prefill_time_ms(prefix_len)
    total_wasted_ms = per_request_ms * n_requests
    print(f"{name:<28} {prefix_len:>8} {n_requests:>10} "
          f"{total_wasted_ms:>20.1f} {total_wasted_ms/1000:>18.2f}")

## 2. Hash-Based Cache Lookup

To check if a prefix has been computed before, we hash its token IDs. The hash must be:
- **Deterministic**: same tokens → same hash
- **Prefix-aware**: cache is keyed by prefix, not full sequence
- **Block-aligned**: in PagedAttention, we cache at block granularity

```
Token sequence: [1024, 531, 8921, 42, 7777, 3344, 9901, ...]
Block size: 4

Block 0 hash: hash([1024, 531, 8921, 42])
Block 1 hash: hash(block_0_hash || [7777, 3344, 9901, ...])
               ↑ chain: each block's hash depends on all previous blocks

This ensures: same tokens in same order → cache hit
```

The chaining is important: block `[531, 8921, 42, 7777]` at position 8 should have a different hash than the same tokens at position 0.

In [ ]:
def hash_block(token_ids: List[int], prev_hash: Optional[str] = None) -> str:
    """
    Hash a block of token IDs, chaining from the previous block's hash.
    This makes the hash position-aware.
    """
    h = hashlib.sha256()
    if prev_hash is not None:
        h.update(prev_hash.encode())
    h.update(bytes(token_ids))  # token IDs as bytes
    return h.hexdigest()[:16]   # truncate to 16 hex chars


def compute_block_hashes(token_ids: List[int], block_size: int) -> List[str]:
    """Compute chained hashes for all complete blocks in a token sequence."""
    hashes = []
    prev_hash = None
    for i in range(0, len(token_ids) - block_size + 1, block_size):
        block = token_ids[i:i + block_size]
        h = hash_block(block, prev_hash)
        hashes.append(h)
        prev_hash = h
    return hashes


# Demonstrate
BLOCK_SIZE = 4
system_prompt = [1024, 531, 8921, 42, 7777, 3344, 9901, 2211]  # 8 tokens = 2 blocks
user_msg_1 = [101, 202, 303, 404]
user_msg_2 = [505, 606, 707, 808]

req1_tokens = system_prompt + user_msg_1
req2_tokens = system_prompt + user_msg_2

hashes_1 = compute_block_hashes(req1_tokens, BLOCK_SIZE)
hashes_2 = compute_block_hashes(req2_tokens, BLOCK_SIZE)

print("Block hash comparison:")
print(f"Block size: {BLOCK_SIZE} tokens")
print()
print(f"{'Block':>6} | {'Tokens':>30} | {'Request 1 hash':>18} | {'Request 2 hash':>18} | {'Match?':>7}")
print("-" * 90)
for i, (h1, h2) in enumerate(zip(hashes_1, hashes_2)):
    start = i * BLOCK_SIZE
    tokens_str = str(req1_tokens[start:start+BLOCK_SIZE])
    match = "✓ HIT" if h1 == h2 else "✗ MISS"
    print(f"{i:>6} | {tokens_str:>30} | {h1:>18} | {h2:>18} | {match:>7}")

print()
print("Blocks 0-1 (system prompt) match → cache hit!")
print("Block 2 differs (different user messages) → cache miss")

## 3. Implementing a Prefix Cache

In [ ]:
@dataclass
class CachedBlock:
    """A cached KV block entry."""
    hash_key: str
    physical_block_id: int
    token_ids: List[int]
    last_accessed: float = field(default_factory=time.time)
    access_count: int = 0


class PrefixCache:
    """
    LRU prefix cache for KV blocks.
    Maps block hashes to physical KV cache block IDs.
    """
    def __init__(self, max_blocks: int, block_size: int):
        self.max_blocks = max_blocks
        self.block_size = block_size
        self.cache: Dict[str, CachedBlock] = {}
        self._next_block_id = 0
        self.hits = 0
        self.misses = 0

    def lookup(self, hash_key: str) -> Optional[int]:
        """Look up a cached block. Returns physical block ID or None."""
        if hash_key in self.cache:
            block = self.cache[hash_key]
            block.last_accessed = time.time()
            block.access_count += 1
            self.hits += 1
            return block.physical_block_id
        self.misses += 1
        return None

    def store(self, hash_key: str, token_ids: List[int]) -> int:
        """Store a computed block in the cache. Evict LRU if full."""
        if hash_key in self.cache:
            return self.cache[hash_key].physical_block_id

        # Evict if full
        if len(self.cache) >= self.max_blocks:
            self._evict_lru()

        block_id = self._next_block_id
        self._next_block_id += 1
        self.cache[hash_key] = CachedBlock(
            hash_key=hash_key,
            physical_block_id=block_id,
            token_ids=token_ids,
        )
        return block_id

    def _evict_lru(self) -> None:
        """Evict the least-recently-used block."""
        lru_key = min(self.cache.keys(), key=lambda k: self.cache[k].last_accessed)
        del self.cache[lru_key]

    def get_prefix_blocks(
        self, token_ids: List[int]
    ) -> Tuple[int, List[int]]:
        """
        Find the longest cached prefix for a token sequence.
        Returns (cached_token_count, list_of_physical_block_ids).
        """
        hashes = compute_block_hashes(token_ids, self.block_size)
        cached_blocks = []

        for h in hashes:
            block_id = self.lookup(h)
            if block_id is None:
                break  # prefix cache ends here
            cached_blocks.append(block_id)

        cached_tokens = len(cached_blocks) * self.block_size
        return cached_tokens, cached_blocks

    def populate(
        self, token_ids: List[int], skip_first_n: int = 0
    ) -> None:
        """Store all blocks of a sequence in the cache."""
        hashes = compute_block_hashes(token_ids, self.block_size)
        for i, h in enumerate(hashes):
            if i * self.block_size >= skip_first_n:
                start = i * self.block_size
                self.store(h, token_ids[start:start + self.block_size])

    @property
    def hit_rate(self) -> float:
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0.0

    def stats(self) -> str:
        return (f"Cache: {len(self.cache)}/{self.max_blocks} blocks, "
                f"hits={self.hits}, misses={self.misses}, hit_rate={self.hit_rate:.1%}")


# Demonstrate prefix cache
cache = PrefixCache(max_blocks=100, block_size=4)

# Simulate a system prompt shared across many requests
SYSTEM_PROMPT = list(range(1000, 1064))  # 64 tokens = 16 blocks

# First request: cache miss, populate cache
cached_count, cached_blocks = cache.get_prefix_blocks(SYSTEM_PROMPT + [5001, 5002, 5003, 5004])
print(f"Request 1 (first ever):")
print(f"  Cached prefix: {cached_count} tokens ({len(cached_blocks)} blocks)")
print(f"  Need to compute: {len(SYSTEM_PROMPT) + 4 - cached_count} tokens")
cache.populate(SYSTEM_PROMPT + [5001, 5002, 5003, 5004])  # store after computing
print(f"  {cache.stats()}")
print()

# Subsequent requests with same system prompt
for i in range(2, 6):
    user_tokens = list(range(6000 + i*10, 6000 + i*10 + 4))
    full_tokens = SYSTEM_PROMPT + user_tokens
    cached_count, cached_blocks = cache.get_prefix_blocks(full_tokens)
    print(f"Request {i}:")
    print(f"  Cached: {cached_count}/{len(full_tokens)} tokens ({cached_count/len(full_tokens)*100:.0f}%)")
    print(f"  Saved: {cached_count} prefill tokens")

print(f"\nFinal: {cache.stats()}")

## 4. TTFT Savings Calculation

In [ ]:
def ttft_ms(
    prompt_tokens: int,
    cached_tokens: int,
    prefill_ms_per_token: float = 0.05,  # ~50 µs/token on A100 for LLaMA-7B
) -> Tuple[float, float]:
    """Returns (ttft_without_cache_ms, ttft_with_cache_ms)."""
    ttft_no_cache = prompt_tokens * prefill_ms_per_token
    ttft_with_cache = (prompt_tokens - cached_tokens) * prefill_ms_per_token
    return ttft_no_cache, max(ttft_with_cache, 1.0)  # min 1ms for overhead


scenarios = [
    ("Short system prompt (200 tok)",   200,  200, 50),
    ("Long system prompt (2000 tok)",  2000, 2000, 100),
    ("RAG 5k-token doc",               5100, 5000, 100),
    ("Multi-turn (3 turns in)",         800,  600,  50),
    ("Few-shot + question",             512,  480,  20),
]

print("TTFT savings with prefix caching:")
print(f"{'Scenario':<35} {'Total':>7} {'Cached':>8} {'TTFT (no cache)':>17} {'TTFT (cached)':>15} {'Speedup':>9}")
print("-" * 95)
for name, total_toks, cached_toks, user_toks in scenarios:
    no_cache_ms, cached_ms = ttft_ms(total_toks, cached_toks)
    speedup = no_cache_ms / cached_ms
    print(f"{name:<35} {total_toks:>7} {cached_toks:>8} {no_cache_ms:>15.0f}ms "
          f"{cached_ms:>13.0f}ms {speedup:>8.1f}×")

## 5. LRU Eviction and Hit Rate Analysis

In [ ]:
import random
random.seed(42)

def simulate_cache_workload(
    n_requests: int,
    n_distinct_prefixes: int,
    prefix_len: int,
    cache_size_blocks: int,
    block_size: int = 16,
    prefix_popularity: str = "uniform",  # "uniform" or "zipf"
) -> dict:
    """
    Simulate prefix cache hit rate under different workloads.
    """
    cache = PrefixCache(max_blocks=cache_size_blocks, block_size=block_size)

    # Generate distinct prefixes
    prefixes = [
        list(range(p * prefix_len, (p + 1) * prefix_len))
        for p in range(n_distinct_prefixes)
    ]

    # Request popularity distribution
    if prefix_popularity == "zipf":
        # Zipf: prefix 0 is most popular, follows 1/k distribution
        weights = [1.0 / (i + 1) for i in range(n_distinct_prefixes)]
        total = sum(weights)
        weights = [w / total for w in weights]
    else:
        weights = [1.0 / n_distinct_prefixes] * n_distinct_prefixes

    saved_tokens = 0
    total_tokens = 0

    for _ in range(n_requests):
        # Choose a prefix
        prefix_idx = random.choices(range(n_distinct_prefixes), weights=weights)[0]
        prefix = prefixes[prefix_idx]
        user_tokens = list(range(90000, 90004))  # 4 unique user tokens
        full_tokens = prefix + user_tokens

        # Check cache
        cached_count, _ = cache.get_prefix_blocks(full_tokens)
        saved_tokens += cached_count
        total_tokens += len(prefix)  # only count prefix tokens (user tokens are always new)

        # After computing, populate cache
        cache.populate(full_tokens)

    return {
        "hit_rate": cache.hit_rate,
        "token_savings": saved_tokens / total_tokens if total_tokens > 0 else 0,
        "cache_size_blocks": cache_size_blocks,
        "n_distinct": n_distinct_prefixes,
    }


print("Cache hit rate vs. cache size and workload (100 requests, prefix=64 tokens, block=16):")
print()

for dist in ["uniform", "zipf"]:
    print(f"Distribution: {dist}")
    print(f"{'Cache blocks':>13} | {'1 prefix':>10} | {'5 prefixes':>12} | {'20 prefixes':>13}")
    print("-" * 55)
    for cache_blocks in [4, 16, 64, 256]:
        results = []
        for n_prefix in [1, 5, 20]:
            r = simulate_cache_workload(
                n_requests=200,
                n_distinct_prefixes=n_prefix,
                prefix_len=64,
                cache_size_blocks=cache_blocks,
                block_size=16,
                prefix_popularity=dist,
            )
            results.append(r["token_savings"])
        print(f"{cache_blocks:>13} | {results[0]:>9.1%} | {results[1]:>11.1%} | {results[2]:>12.1%}")
    print()

## 6. When Prefix Caching Helps (and When It Doesn't)

In [ ]:
situations = [
    (
        "High benefit",
        [
            "Same system prompt across all requests (chatbot)",
            "Same document context for many questions (RAG)",
            "Same few-shot examples for all requests",
            "Multi-turn conversation (each turn reuses previous)",
            "Code editor with the same large file context",
        ]
    ),
    (
        "Low benefit",
        [
            "Unique prompts (no shared prefix)",
            "Short prompts (little to cache)",
            "Dynamic system prompts (user-customized per-request)",
            "One-shot API calls without recurring patterns",
        ]
    ),
    (
        "Gotchas",
        [
            "Inserting user name/time in system prompt BREAKS caching",
            "'Current date: 2024-01-15' in every prompt → daily cache invalidation",
            "Shuffling few-shot examples → different hash even if same examples",
            "Prefix must be at start of prompt — middle insertions don't cache",
        ]
    ),
]

for category, items in situations:
    print(f"\n{category}:")
    for item in items:
        marker = "✓" if category == "High benefit" else ("✗" if category == "Low benefit" else "⚠")
        print(f"  {marker} {item}")

## 7. API-Level Prompt Caching

Major providers offer explicit prefix caching APIs:

### Anthropic (Claude)
Mark sections of your prompt with `cache_control: {"type": "ephemeral"}`:
```python
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": long_document,  # 50k tokens
                "cache_control": {"type": "ephemeral"}  # ← cache this
            },
            {"type": "text", "text": question}  # not cached
        ]
    }
]
```
- Cache TTL: 5 minutes
- Minimum cacheable: 1024 tokens
- Pricing: cached reads cost 10% of normal input price

### OpenAI
Automatic prefix caching (no explicit markup needed):
- Prompts > 1024 tokens are automatically cached
- Cache hit: 50% discount on cached input tokens
- TTL: not publicly specified

### Google (Gemini)
Explicit context caching via API:
- `caches.create(model, contents)` → cache ID
- Reference cache ID in subsequent requests
- Billed separately: storage cost + reduced per-request cost

In [ ]:
# Cost savings model for API-level caching

def compute_cost_savings(
    prefix_tokens: int,
    n_requests: int,
    input_cost_per_mtok: float = 3.0,    # $/M tokens (Claude Sonnet)
    cached_cost_per_mtok: float = 0.30,  # 90% discount on cache reads
    cache_write_cost_per_mtok: float = 3.75,  # 25% premium on cache writes
) -> dict:
    """Model cost savings from prefix caching."""
    # Without caching: pay full price for prefix on every request
    cost_no_cache = n_requests * prefix_tokens / 1e6 * input_cost_per_mtok

    # With caching: pay write price once, read price for all others
    cache_write_cost = prefix_tokens / 1e6 * cache_write_cost_per_mtok
    cache_read_cost = (n_requests - 1) * prefix_tokens / 1e6 * cached_cost_per_mtok
    cost_with_cache = cache_write_cost + cache_read_cost

    return {
        "cost_no_cache": cost_no_cache,
        "cost_with_cache": cost_with_cache,
        "savings": cost_no_cache - cost_with_cache,
        "savings_pct": (cost_no_cache - cost_with_cache) / cost_no_cache * 100,
    }


print("Cost savings from prefix caching (Claude Sonnet pricing):")
print(f"{'Scenario':<35} {'Prefix':>8} {'Requests':>10} {'Without':>10} {'With':>10} {'Savings':>10} {'%':>6}")
print("-" * 95)

scenarios = [
    ("System prompt, 100 req/day",      500,   100),
    ("System prompt, 10k req/day",      500, 10000),
    ("RAG doc (5k tok), 100 req",      5000,   100),
    ("RAG doc (5k tok), 10k req",      5000, 10000),
    ("Book analysis (50k tok)",        50000,   50),
]

for name, prefix, n_req in scenarios:
    s = compute_cost_savings(prefix, n_req)
    print(f"{name:<35} {prefix:>8,} {n_req:>10,} "
          f"${s['cost_no_cache']:>8.2f} ${s['cost_with_cache']:>8.2f} "
          f"${s['savings']:>8.2f} {s['savings_pct']:>5.0f}%")

## 8. Structuring Prompts for Maximum Cache Utilization

In [ ]:
print("Prompt structure guidelines for maximum cache hit rate:")
print()

print("GOOD: Static content first, dynamic content last")
print("""
  [SYSTEM PROMPT - static, never changes]         ← cached
  [FEW-SHOT EXAMPLES - static, in fixed order]    ← cached
  [DOCUMENT CONTEXT - static for session]         ← cached
  [CONVERSATION HISTORY - grows each turn]        ← partially cached
  [CURRENT USER MESSAGE - always new]             ← never cached
""")

print("BAD: Dynamic content mixed into static sections")
print("""
  [SYSTEM: 'Today is {date}. You are...'  ]   ← BREAKS caching (date changes daily)
  [SYSTEM: 'Hello {username}! You are...']    ← BREAKS caching (unique per user)
  [SHUFFLED few-shot examples each request]  ← BREAKS caching (different hash)
  [PERSONALIZATION injected mid-prompt]      ← BREAKS caching
""")

print("RULE: Cache breaks at the first token that differs between requests.")
print("      Everything before that break is cacheable.")
print("      Everything after that break is not cached.")
print()

# Simulate cache hit rate for good vs bad prompt structure
def prompt_cache_efficiency(static_tokens: int, dynamic_tokens: int,
                             dynamic_position: str = "end") -> dict:
    total = static_tokens + dynamic_tokens
    if dynamic_position == "end":
        cacheable = static_tokens
    elif dynamic_position == "start":
        cacheable = 0  # break at position 0
    else:  # middle
        cacheable = static_tokens // 2  # rough estimate

    return {
        "total_tokens": total,
        "cacheable_tokens": cacheable,
        "cache_efficiency": cacheable / total,
    }


print("Cache efficiency by prompt structure:")
cases = [
    ("Dynamic at end (optimal)",   2000, 50, "end"),
    ("Dynamic in middle",          2000, 50, "middle"),
    ("Dynamic at start (worst)",   2000, 50, "start"),
]
for name, static, dynamic, pos in cases:
    e = prompt_cache_efficiency(static, dynamic, pos)
    print(f"  {name:<30}: {e['cache_efficiency']:>6.1%} cacheable "
          f"({e['cacheable_tokens']}/{e['total_tokens']} tokens)")

## Summary

```
Problem: Repeated prefix computation across requests wastes compute

Solution: Prefix caching
  - Hash each KV block (chained for position-awareness)
  - Cache hit → skip prefill for those tokens
  - LRU eviction when cache is full

Benefits:
  - TTFT: 10–100× faster when prefix is cached
  - Cost: 60–90% reduction for prefix tokens (API pricing)
  - Throughput: more GPU capacity for new requests

Key rules:
  - Static content first, dynamic content last
  - Never inject per-request data into cacheable prefixes
  - Cache breaks at the first differing token
```

**Next:** Notebook 09 covers quantization — reducing model precision from FP16 to INT8/INT4 to cut memory and increase inference speed, with the quality tradeoff curves.